## Exemplo de código

In [1]:
class ProdutoImportado:

    def __init__(self, ncm, descricao, valor_cif, aliquota):
        self.ncm = ncm
        self.descricao = descricao
        self.valor_cif = valor_cif
        self.aliquota = aliquota

    def tarifa_importacao(self):
        return self.valor_cif * self.aliquota

    def valor_final(self):
        print('Calculando valor final do produto importado sem demais impostos...')
        return self.valor_cif + self.tarifa_importacao()
    

class ProdutoIndustrial(ProdutoImportado):

    def __init__(self, ncm, descricao, valor_cif, aliquota_importacao, aliquota_ipi, aliquota_pis, aliquota_cofins):
        super().__init__(ncm, descricao, valor_cif, aliquota_importacao)
        self.aliquota_ipi = aliquota_ipi
        self.aliquota_pis = aliquota_pis
        self.aliquota_cofins = aliquota_cofins

    def valor_final(self):
        valor_com_imposto_importacao = super().valor_final()
        print('Calculando valor final do produto industrial com todos os impostos...')

        valor_ipi = valor_com_imposto_importacao * self.aliquota_ipi
        valor_pis = valor_com_imposto_importacao * self.aliquota_pis
        valor_cofins = valor_com_imposto_importacao * self.aliquota_cofins
        return valor_com_imposto_importacao + valor_ipi + valor_pis + valor_cofins

In [2]:
notebook = ProdutoImportado("12345678", "Produto Importado", 100.0, 0.2)
print(f"Valor final do produto importado: {notebook.valor_final()}")

Calculando valor final do produto importado sem demais impostos...
Valor final do produto importado: 120.0


In [3]:
notebook_industrial = ProdutoIndustrial("87654321", "Produto Industrial", 100.0, 0.2, 0.1, 0.05, 0.03)
print(f"Valor final do produto industrial: {notebook_industrial.valor_final()}")

Calculando valor final do produto importado sem demais impostos...
Calculando valor final do produto industrial com todos os impostos...
Valor final do produto industrial: 141.6


## Carregamento de base

In [ ]:
##Baixar os dados de importação do ano de 2026 a partir do link https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/EXP_2026.csv e carregar em dados/comex/movimentacoes usando apenas bibliotecas padrão do Python. O arquivo CSV é separado por ponto e vírgula e codificado em latin-1.
import csv
import urllib.request
import ssl
import os
import io

def baixar_dados_importacao(url, caminho_destino):
    try:
        # Cria um contexto SSL que não verifica certificados (para contornar o erro SSL)
        contexto_ssl = ssl.create_default_context()
        contexto_ssl.check_hostname = False
        contexto_ssl.verify_mode = ssl.CERT_NONE
        
        with urllib.request.urlopen(url, context=contexto_ssl) as response:
            csv_data = response.read().decode('latin-1')
            os.makedirs(os.path.dirname(caminho_destino), exist_ok=True)
            with open(caminho_destino, 'w', encoding='latin-1') as file:
                file.write(csv_data)
        print(f'Dados de importação baixados com sucesso e salvos em {caminho_destino}')
    except Exception as e:
        print(f'Erro ao baixar os dados: {e}')

url = 'https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/IMP_2025.csv'
caminho_destino = '../dados/comex/movimentacoes/IMP_2026.csv'  # Caminho corrigido
baixar_dados_importacao(url, caminho_destino)

Dados de importação baixados com sucesso e salvos em dados/comex/movimentacoes/EXP_2026.csv


In [ ]:
# Alternativa usando requests (mais robusta) - instale com: pip install requests
# import requests
# import os

# def baixar_dados_importacao_requests(url, caminho_destino):
#     try:
#         # Desabilita avisos SSL e permite downloads sem verificação
#         import urllib3
#         urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
#         
#         response = requests.get(url, verify=False, stream=True)
#         response.encoding = 'latin-1'
#         
#         os.makedirs(os.path.dirname(caminho_destino), exist_ok=True)
#         
#         with open(caminho_destino, 'w', encoding='latin-1') as file:
#             file.write(response.text)
#             
#         print(f'Dados de importação baixados com sucesso e salvos em {caminho_destino}')
#     except Exception as e:
#         print(f'Erro ao baixar os dados: {e}')

# url = 'https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/EXP_2026.csv'
# caminho_destino = 'dados/comex/movimentacoes/EXP_2026.csv'
# baixar_dados_importacao_requests(url, caminho_destino)

In [3]:
# Verificando dados baixados e carregando uma amostra
import pandas as pd

# Carregar algumas linhas do arquivo para verificar a estrutura
df_sample = pd.read_csv('../dados/comex/movimentacoes/EXP_2026.csv', 
                       sep=';', encoding='latin-1', nrows=5)

print("Estrutura dos dados:")
print(df_sample.head())
print(f"\nColunas disponíveis: {list(df_sample.columns)}")
print(f"Forma da amostra: {df_sample.shape}")

# Verificar se o arquivo existe e seu tamanho
import os
arquivo = '../dados/comex/movimentacoes/EXP_2026.csv'
if os.path.exists(arquivo):
    tamanho = os.path.getsize(arquivo) / (1024*1024)  # MB
    print(f"✅ Arquivo baixado com sucesso: {tamanho:.2f} MB")
else:
    print("❌ Arquivo não encontrado")

Estrutura dos dados:
   CO_ANO  CO_MES    CO_NCM  CO_UNID  CO_PAIS SG_UF_NCM  CO_VIA   CO_URF  \
0    2026       1  84672999       11      580        PR       1   817800   
1    2026       2  39235000       10      586        RS       7   917500   
2    2026       2  85444200       10      158        RJ       4   817600   
3    2026       2  84821010       11       63        SP       4   817700   
4    2026       2  85443000       10      845        MG       7  1017701   

   QT_ESTAT  KG_LIQUIDO  VL_FOB  
0      2584       12894  140073  
1       284         284    2454  
2         0           0     971  
3         5           8    1617  
4         9           9     346  

Colunas disponíveis: ['CO_ANO', 'CO_MES', 'CO_NCM', 'CO_UNID', 'CO_PAIS', 'SG_UF_NCM', 'CO_VIA', 'CO_URF', 'QT_ESTAT', 'KG_LIQUIDO', 'VL_FOB']
Forma da amostra: (5, 11)
✅ Arquivo baixado com sucesso: 15.72 MB
